# Khai phá Tập Phổ Biến bằng Thuật toán GenMax

Đây là tệp Notebook trình diễn toàn bộ chức năng của kho mã nguồn. Dự án hoàn toàn được xây dựng bằng **ngôn ngữ Julia**, bao gồm code lõi tự viết (từ cấu trúc dữ liệu Vertical Database, biểu diễn BitSet, và sinh tập tối đại MFI), cũng như quy trình sinh luật phục vụ bài toán Market Basket Analysis.

---

## 1. Khởi tạo Không gian Môi trường & Core Modules

Chúng ta sẽ import các cấu trúc chính của thuật toán, định sẵn biến `Random.seed!(42)` nhằm mục đích tái sản xuất cố định cấu trúc mỗi khi chạy test.

In [1]:
using Random
Random.seed!(42)

include("../src/structures.jl")
include("../src/algorithm/genmax.jl")
include("../src/utils.jl")

using .Structures
using .GenMaxAlgo
using .Utils

println("Setup Môi trường Hoàn tất!")

Setup Môi trường Hoàn tất!


## 2. Trình diễn Thuật toán trên Dữ liệu Cơ sở (Toy Dataset)

Để chứng minh thuật toán hoạt động chính xác với cơ chế theo tài liệu khoa học, ta giả lập tệp dữ liệu `toy.txt` gồm các giao dịch A B C D ngắn. Qua đó quan sát cách mà tổ hợp Diffset và thuật toán quay lui (backtracking) tìm ra Maximal Frequent Itemsets (MFI).

In [2]:
toy_dir = "../data/toy/"
toy_path = joinpath(toy_dir, "toy.txt")

# Đọc dữ liệu nhỏ
toy_data = read_spmf_file(toy_path)
println("\n* Dữ liệu các giao dịch (Transactions):\n")
for (i, t) in enumerate(toy_data)
    println("  TID $i : ", Base.join(t, " "))
end

minsup_toy = 3
println("\n* Chạy GenMax với Minsup Cứng (Absolute) = $minsup_toy...")
mfi_toy = genmax(toy_data, minsup_toy)

println("\n* Có ", length(mfi_toy), " tập phổ biến Tối Đại (MFI) được tìm ra:")
for (itemset, sup) in mfi_toy
    println("  Itemset: { ", Base.join(itemset, ", "), " }  | Support: ", sup)
end


* Dữ liệu các giao dịch (Transactions):

  TID 1 : A B C D
  TID 2 : A B C
  TID 3 : A B C F
  TID 4 : D E F
  TID 5 : D E
  TID 6 : D E

* Chạy GenMax với Minsup Cứng (Absolute) = 3...

* Có 2 tập phổ biến Tối Đại (MFI) được tìm ra:
  Itemset: { D, E }  | Support: 3
  Itemset: { A, B, C }  | Support: 3


## 3. Đánh giá Hiệu năng cực đỉnh trên Benchmark (Performance)
Kiểm thử khả năng tối ưu RAM và xử lý khối lượng lớn bằng file `chess.dat` (>3.100 giao dịch). Lệnh `@time` của Julia sẽ xuất ra bằng chứng cho thấy thuật toán có hiệu năng thực thi cực nhanh.

In [3]:
benchmark_path = "../data/benchmark/chess.dat"
println("* Đang đọc dữ liệu Benchmark: ", benchmark_path)
chess_data = read_spmf_file(benchmark_path)
println("  Số lượng hóa đơn: ", length(chess_data))

minsup_chess = 0.8  # Tìm hóa đơn tần suất mua lên tới 80%
println("\n* Bắt đầu đo lường hiệu năng xử lý...")
mfi_chess = @time genmax(chess_data, minsup_chess)

println("\n> Tìm được ", length(mfi_chess), " Tập Tối Đại cực dài!!")
println("> Trích xuất Top 5 tập đắt giá nhất:")
for i in 1:min(5, length(mfi_chess))
    println("  ", Base.join(mfi_chess[i][1], ", "), " (Xuất hiện: ", mfi_chess[i][2], " lần)")
end

* Đang đọc dữ liệu Benchmark: ../data/benchmark/chess.dat
  Số lượng hóa đơn: 3196

* Bắt đầu đo lường hiệu năng xử lý...
  0.090836 seconds (112.02 k allocations: 7.455 MiB, 92.32% compilation time)

> Tìm được 228 Tập Tối Đại cực dài!!
> Trích xuất Top 5 tập đắt giá nhất:
  46 (Xuất hiện: 2556 lần)
  44, 58, 60 (Xuất hiện: 2564 lần)
  29, 40, 44, 52, 58 (Xuất hiện: 2559 lần)
  29, 40, 52, 58, 60, 64 (Xuất hiện: 2569 lần)
  29, 42, 5 (Xuất hiện: 2556 lần)


## 4. Cơ chế Toán học Sinh luật (Association Rules Mechanism)
Từ dữ liệu Toy Dataset, chúng ta sẽ cho thuật toán phân lớp MFI ra 2 chuỗi Tiền Đề (Antecedent) và Hậu Qủa (Consequent) nhằm sinh ra các luật `X -> Y`. Chúng ta cho hiển thị cả Support, Confidence và Lift để dễ dàng tự lấy máy tính bấm đối chiếu độ chuẩn xác.

In [4]:
"""Hàm tính lại support nhánh cho Toy Data"""
function get_support(itemset, db)
    count = 0
    for tx in db
        if issubset(itemset, tx)
            count += 1
        end
    end
    return count
end

function show_toy_rules(data, mfi_results, minconf)
    rules = []
    total_tx = length(data)
    
    for (mfi, sup_val) in mfi_results
        if length(mfi) < 2 continue end
        for i in 1:length(mfi)
            con = [mfi[i]]
            ant = setdiff(mfi, con)
            
            sup_A = get_support(ant, data)
            sup_B = get_support(con, data)
            
            conf = sup_val / sup_A
            if conf >= minconf
                lift = (sup_val / total_tx) / ((sup_A / total_tx) * (sup_B / total_tx))
                push!(rules, (ant, con, conf, lift))
            end
        end
    end
    sort!(rules, by=x->x[4], rev=true)
    return rules
end

rules = show_toy_rules(toy_data, mfi_toy, 0.7)
println("--- LUẬT TRÊN TOY DATA ---")
for (i, r) in enumerate(rules)
    println("Rule $i: {", Base.join(r[1],", "), "} -> {", Base.join(r[2],","),
            "} | Conf: ", round(r[3], digits=2), " | Lift: ", round(r[4], digits=2))
end

--- LUẬT TRÊN TOY DATA ---
Rule 1: {B, C} -> {A} | Conf: 1.0 | Lift: 2.0
Rule 2: {A, C} -> {B} | Conf: 1.0 | Lift: 2.0
Rule 3: {A, B} -> {C} | Conf: 1.0 | Lift: 2.0
Rule 4: {E} -> {D} | Conf: 1.0 | Lift: 1.5
Rule 5: {D} -> {E} | Conf: 0.75 | Lift: 1.5


## 5. Ứng dụng Thực chiến Giỏ hàng (Market Basket - Retail)
Sử dụng module sinh luật mạnh mẽ từ `src/market_basket.jl` để thực thi quét toàn bộ cửa hàng bán lẻ ảo (dataset `retail.dat`). Chúng ta sẽ tìm tổ hợp luật mua sắm thông minh.

In [5]:
module RetailDemo
    println("* Đang tải module phân tích giỏ hàng...")
    include("../src/market_basket.jl")
    retail_path = "../data/application/retail.dat"
    
    # Chạy API Market Basket Analysis đóng gói.
    # Xét mặt hàng mua lặp lại >= 5% và tỷ lệ dự đoán >= 50%.
    market_basket_analysis(retail_path, 0.05, 0.50)
end

* Đang tải module phân tích giỏ hàng...
MARKET BASKET ANALYSIS (PHÂN TÍCH GIỎ HÀNG)
Dữ liệu: retail.dat
Minsup:  0.05
Minconf: 0.5
[1] Đang chạy GenMax để tìm Maximal Frequent Itemsets (MFI)...
    Tìm được 4 MFI.
[2] Đang trích xuất tất cả Frequent Itemsets từ kết quả MFI...
    Tổng số Frequent Itemsets sau khi bung tổ hợp: 16
[3] Đang tính chính xác giá trị support cho từng Frequent Itemset...
[4] Đang sinh luật kết hợp và tính Lift...
    Tìm được 14 luật thỏa mãn minconf.

=== TOP 10 LUẬT KẾT HỢP (THEO LIFT) ===

Vế Trái (X)                    => Vế Phải (Y)                    | Sup    | Conf   | Lift  
-----------------------------------------------------------------------------------------------
["42", "49"]                   => ["40"]                         | 7366   | 0.82   | 1.42  
["40", "42"]                   => ["49"]                         | 7366   | 0.65   | 1.35  
["33", "40"]                   => ["49"]                         | 5402   | 0.64   | 1.34  
["39", "49"]

Main.RetailDemo

## 6. Bộ Kiểm Thử Tự Động Toàn Diện (Unit Tests Pipeline)
Luôn phải chứng minh mọi thuật toán không mắc lỗi cơ bản qua thời gian. Dự án xây dựng sẵn bộ test chuẩn với package Test của Julia.

In [6]:
println("[Hệ thống] Đang chạy kiểm thử tích hợp (Integration Tests)...\n")
include("../test/runtests.jl")
println("\n[Hệ thống] Đạt tiêu chuẩn chất lượng (100% Passed)!")

[Hệ thống] Đang chạy kiểm thử tích hợp (Integration Tests)...

Test Summary: | Pass  Total  Time
All Tests     |   15     15  6.4s

[Hệ thống] Đạt tiêu chuẩn chất lượng (100% Passed)!
